260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1 - VOLUSEG

/ 260605, run ON HPC

In [1]:
print('do smth bro')

do smth bro


In [2]:
# set up
import os
import pprint
import voluseg
import h5py
import numpy as np


from skimage import io
import matplotlib.pyplot as plt
import os, glob

import tifffile as tif
import warnings

#check for updates
#voluseg.update()


# ===============================

proj_ID = 'hcrt-trpv1_huc-h2b-g8m_csn_120min_May26'
expt_ID = '260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1'
frames_per_volume = 40
seconds_per_volume = 1
z_range = 250
binning = 1
pixel_size = 1.52
# ===============================


In [3]:
import findspark
findspark.init()
import pyspark
from pyspark.sql import SparkSession
#sparkSession3 = SparkSession.newSession

# spark specs from Altyn
spark = SparkSession.builder \
     .master("local[*]") \
     .config("spark.driver.maxResultSize", "0") \
     .config("spark.executor.memory", "80g") \
     .config("spark.driver.memory", "80g") \
     .config("spark.memory.offHeap.enabled", True) \
     .config("spark.memory.offHeap.size","40g")   \
     .appName("sampleCodeForReference") \
     .getOrCreate()

In [4]:
from pyspark.sql import SparkSession
# STOP whatever is running
try:
    spark.stop()
except:
    pass


spark = (
    SparkSession.builder \
    .master("local[24]")
    .config("spark.executor.instances", "8")          # 8 JVMs
    .config("spark.executor.cores", "5")              # 2 tasks per executor (caps concurrency)
    .config("spark.driver.maxResultSize", "0")
    .config("spark.executor.memory", "80g")           # per-executor heap
    .config("spark.driver.memory", "80g")
    .config("spark.memory.offHeap.enabled", True)
    .config("spark.memory.offHeap.size","40g")
    .config("spark.default.parallelism", "9")
    .config("spark.sql.shuffle.partitions", "96")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 17:18:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# set and save parameters
#fname = '/ssd-pool/Altyn/voluseg/raw_datasets_voluseg/20231025_HuC_H2B_GCaMP7f_MT/fish1/20231025_HuC_H2B_GCaMP7f_fish1_920nm_250um_1_5sec_per_vol_25slices_per_vol_bin2x_15minE3_15minDMSO_30min10uMMelatonin_30min_blue_flash_every10min_FullRun_1.tif'
#base ='/mnt/storageServer/Yun/LightSheet/'
base = '/resnick/groups/Proberlab/yun/lightsheet/'                                                                

parameters0 = voluseg.parameter_dictionary()
#parameters0['dir_ants'] = '/ssd-pool/yun/voluseg/voluseg_setup/install/bin'
parameters0['dir_ants'] = base + 'voluseg_setup/install/bin'

parameters0['dir_input'] =  base + proj_ID + '/' + expt_ID + '/' + 'input' 
parameters0['dir_output'] = base + proj_ID + '/' + expt_ID + '/' + 'output' 

### Voluseg Parameters: (modified based on experiments!)

parameters0['registration'] = 'high'
parameters0['diam_cell'] = 5.0
parameters0['f_volume'] = 1 / frames_per_volume
parameters0['t_section'] = seconds_per_volume / frames_per_volume
parameters0['ds'] = 1 # 1=means no downsampling; 2=downsampling by 2

parameters0['res_x'] = pixel_size * binning # pixel size after 2024-7-1
parameters0['res_y'] = pixel_size * binning

parameters0['res_z'] = z_range / frames_per_volume # z-range (µm) / number of slices; 300/30 = 10

#parameters0['timepoints'] = 1000 # when it is set to zero it means we provide the entire dataset for the segmentation
#parameters0['']
parameters0['parallel_clean']  = False   # ***critical*** avoid accumulator in step5

voluseg.step0_process_parameters(parameters0)

Parameter file exists at: /resnick/groups/Proberlab/yun/lightsheet/hcrt-trpv1_huc-h2b-g8m_csn_120min_May26/260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/output/parameters.json.
Loading parameters from file.
Parameters file successfully loaded and validated.


{'detrending': 'standard',
 'registration': 'high',
 'registration_restrict': '',
 'diam_cell': 5.0,
 'dir_ants': '/resnick/groups/Proberlab/yun/lightsheet/voluseg_setup/install/bin',
 'dir_input': '/resnick/groups/Proberlab/yun/lightsheet/hcrt-trpv1_huc-h2b-g8m_csn_120min_May26/260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/input',
 'dir_output': '/resnick/groups/Proberlab/yun/lightsheet/hcrt-trpv1_huc-h2b-g8m_csn_120min_May26/260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/output',
 'dir_transform': '',
 'ds': 1,
 'planes_pad': 0,
 'planes_packed': False,
 'parallel_clean': False,
 'parallel_volume': True,
 'save_volume': False,
 'type_timepoints': 'dff',
 'type_mask': 'geomean',
 'timepoints': 1000,
 'f_hipass': 0.0,
 'f_volume': 0.025,
 'n_cells_block': 316,
 'n_colors': 1,
 'res_x': 1.52,
 'res_y': 1.52,
 'res_z': 6.25,
 't_baseline': 300,
 't_section': 0.025,
 'thr_mask': 0.5,
 'volume_fullnames_input': array(['/resnick/groups/Proberlab/yun/lightsheet/hcrt-trpv1_huc-h2b-g8m_csn_120min

In [6]:
# load and print parameters
filename_parameters = os.path.join(parameters0['dir_output'], 'parameters.json')
parameters = voluseg.load_parameters(filename_parameters)

Parameters file successfully loaded and validated.


In [7]:
print("process volumes.")
voluseg.step1_process_volumes(parameters)

process volumes.


In [8]:
print("align volumes.")
voluseg.step2_align_volumes(parameters)

align volumes.


In [9]:
print("mask volumes.")
voluseg.step3_mask_volumes(parameters)

mask volumes.


In [10]:
print("detect cells.")
voluseg.step4_detect_cells(parameters)

detect cells.
number of blocks, total: 1443.
number of blocks, remaining: 23.


data loading: 1.2 minutes.                                          (0 + 9) / 9]

voxel similarity: 0.0 minutes.

(1.0, 517)
/home/ychiu/miniconda3/envs/voluseg/lib/python3.10/site-packages/sklearn/cluster/_agglomerative.py:325: UserWarning: the number of connected components of the connectivity matrix is 3 > 1. Completing it to avoid stopping the tree early.
  connectivity, n_connected_components = _fix_connectivity(
cell initialization: 0.0 minutes.

(0, 5.858555390885965, 1.811674554047686)
(1, 0.8801311519769043, 1.4580345185907235)
(2, 0.9054090566936581, 0.046675646264242227)
data loading: 1.4 minutes.

data loading: 1.4 minutes.

(3, 0.9089250846342891, 0.004286189963620793)
(4, 0.9111381620101998, 0.0027621097618911695)
data loading: 1.5 minutes.

data loading: 1.5 minutes.

data loading: 1.5 minutes.

data loading: 1.6 minutes.

voxel similarity: 0.0 minutes.

(1.0, 454)
/home/ychiu/miniconda3/envs/voluseg/lib/python3.10/site-packages/sklearn/cluster/_agglomerative.py:325: Use

In [11]:
print("clean cells.")
voluseg.step5_clean_cells(parameters)

clean cells.
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
nans (to be removed): 3


/home/ychiu/miniconda3/envs/voluseg/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/ychiu/miniconda3/envs/voluseg/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Computing baseline in serial mode... done.


In [12]:
# STOP whatever is running
try:
    spark.stop()
except:
    pass

import gc, time
gc.collect(); time.sleep(2)


# 